# Otros datasets (Anexo B)

Notebook que genera los archivos necesarios para el Anexo B de la tesis:
highlights JSON, boxplot SVGs y scatter SVGs para los 10 datasets
no incorporados en el cuerpo principal.

Archivos generados:
- **JSONs** en `docs/data/`: `{dataset}-r2-highlights.json`
- **SVGs** en `docs/img/`: `{dataset}-scatter.svg`, `{dataset}-r2-boxplot.svg`, `{dataset}-accuracy-boxplot.svg`

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

# Must be set before importing matplotlib for deterministic SVG output
os.environ["SOURCE_DATE_EPOCH"] = "0"

In [ ]:
import json
import logging
import pickle
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import HTML
from IPython.display import display as ipy_display
from matplotlib.pyplot import close
from sklearn.exceptions import InconsistentVersionWarning

from fkdc.config import _get_run_seeds, main_seed
from fkdc.datasets import found_datasets, synth_datasets
from fkdc.viz import (
    apply_hatching,
    boxplot,
    default_palette,
    get_highlights,
    load_infos,
    parse_basic_info,
)

In [ ]:
warnings.filterwarnings("ignore", category=InconsistentVersionWarning)

# ASCII minus in SVG tick labels (U+2212 breaks Typst SVG parsing)
plt.rcParams["axes.unicode_minus"] = False
# Deterministic SVG element IDs
plt.rcParams["svg.hashsalt"] = "fkdc"

logging.basicConfig(
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    level=logging.INFO,
    handlers=[logging.StreamHandler()],
)
logger = logging.getLogger("otros-datasets")

In [ ]:
THUMB_WIDTH = 200


def show_static(*paths, cols=3, width=None):
    w = width or THUMB_WIDTH
    items = list(paths)
    rows_html = ""
    for i in range(0, len(items), cols):
        chunk = items[i : i + cols]
        tds = "".join(
            f'<td style="text-align:center;padding:4px">'
            f'<img src="{p}" width="{w}"><br>'
            f"<small>{p.name}</small></td>"
            for p in chunk
        )
        rows_html += f"<tr>{tds}</tr>"
    ipy_display(HTML(f"<table>{rows_html}</table>"))


_generated = []


def save_fig(fig, fpath, **savefig_kw):
    savefig_kw.setdefault("bbox_inches", "tight")
    fig.savefig(fpath, **savefig_kw)
    logger.info(f"Wrote {fpath}")
    _generated.append(fpath)
    close(fig)
    plt.close()


def show_generated(cols=5):
    global _generated
    if _generated:
        show_static(*_generated, cols=cols, width=THUMB_WIDTH)
        _generated = []

In [ ]:
run_seeds = _get_run_seeds()
plotting_seed = run_seeds[0]

root_dir = Path("/Users/gonzalo/Git/fkdc")
run_dir = root_dir / "sandbox/v5/infos"
datasets_dir = run_dir / "../datasets"
img_dir = root_dir / "docs/img"
data_dir = root_dir / "docs/data"
for d in [data_dir, img_dir]:
    d.mkdir(exist_ok=True)

cache_path = root_dir / ".cache/infos_bi.pkl"
cache_path.parent.mkdir(exist_ok=True)
if cache_path.exists():
    logger.info(f"Loading cached infos+bi from {cache_path}")
    with open(cache_path, "rb") as fp:
        infos, bi = pickle.load(fp)
else:
    logger.info("Loading infos from disk (first run, will cache)...")
    infos = load_infos(run_dir)
    bi = parse_basic_info(infos, main_seed)
    with open(cache_path, "wb") as fp:
        pickle.dump((infos, bi), fp)
    logger.info(f"Cached infos+bi to {cache_path}")

In [ ]:
# The 10 datasets not in the main body
anexo_datasets = [
    # 15D (3D + 12 noise dims)
    "pionono_12",
    "eslabones_12",
    "helices_12",
    "hueveras_12",
    # Multiclass real-world
    "iris",
    "vino",
    "pinguinos",
    "anteojos",
    # High-dimensional
    "digitos",
    "mnist",
]

## Generar highlights JSON, scatter SVGs y boxplot SVGs

Genera para cada dataset:
- `{dataset}-r2-highlights.json`
- `{dataset}-scatter.svg`
- `{dataset}-r2-boxplot.svg`
- `{dataset}-accuracy-boxplot.svg`

In [ ]:
# Highlights JSON
for dataset in anexo_datasets:
    hl = get_highlights(dataset, by="r2", info=bi)
    hl["summary"] = hl["summary"].round(4).to_csv()
    fname = f"{dataset}-r2-highlights.json"
    fpath = data_dir / fname
    with open(fpath, "w") as fp:
        json.dump(hl, fp, indent=4)
    logger.info(f"Wrote {fpath}")

In [ ]:
# Scatter SVGs
for dataset in anexo_datasets:
    if dataset in synth_datasets:
        fpath_ds = datasets_dir / f"{dataset}-{plotting_seed}.pkl"
    elif dataset in found_datasets:
        fpath_ds = datasets_dir / f"{dataset}.pkl"
    else:
        raise ValueError(f"Unknown dataset: {dataset}")
    with open(fpath_ds, "rb") as fp:
        ds = pickle.load(fp)
    fig, ax = plt.subplots(layout="tight")
    ds.scatter(ax=ax)
    ax.set_title(dataset)
    fpath = img_dir / f"{dataset}-scatter.svg"
    save_fig(fig, fpath)
show_generated(cols=5)

In [ ]:
# Boxplot SVGs (R² and accuracy)
for dataset in anexo_datasets:
    for metric in ["r2", "accuracy"]:
        fig, ax = plt.subplots(layout="tight")
        boxplot(dataset, metric, info=bi, ax=ax, palette=default_palette)
        fpath = img_dir / f"{dataset}-{metric}-boxplot.svg"
        save_fig(fig, fpath)
show_generated(cols=5)

## Exploración: resumen comparativo

Tabla con el mejor clasificador (por mediana de R²) para cada dataset del anexo,
y la diferencia entre la variante Fermat y la euclídea.

In [ ]:
rows = []
for dataset in anexo_datasets:
    sub = bi[bi.dataset.eq(dataset)].dropna(subset="r2")
    medians = sub.groupby("clf")["r2"].median().sort_values(ascending=False)
    best_clf = medians.index[0]
    best_r2 = medians.iloc[0]

    # Fermat vs Euclidean deltas for paired classifiers
    pairs = [("fkdc", "kdc"), ("fkn", "kn"), ("slr", "lr")]
    deltas = {}
    for fermat, eucl in pairs:
        if fermat in medians.index and eucl in medians.index:
            deltas[f"{fermat}-{eucl}"] = medians[fermat] - medians[eucl]

    rows.append(
        {
            "dataset": dataset,
            "best": best_clf,
            "R² (med)": round(best_r2, 4),
            **{k: round(v, 4) for k, v in deltas.items()},
        }
    )

summary_df = pd.DataFrame(rows).set_index("dataset")
summary_df

## Exploración: distribución de R² por dataset

Boxplots agrupados mostrando todos los datasets del anexo en una sola figura,
filtrado a los clasificadores basados en densidad (fkdc, kdc, fkn, kn).

In [ ]:
density_clfs = ["fkdc", "kdc", "fkn", "kn"]
sub = bi[bi.dataset.isin(anexo_datasets) & bi.clf.isin(density_clfs)].dropna(
    subset="r2"
)

fig, ax = plt.subplots(figsize=(14, 5), layout="tight")
sns.boxplot(
    sub,
    x="dataset",
    y="r2",
    hue="clf",
    palette={c: default_palette[c] for c in density_clfs},
    ax=ax,
)
apply_hatching(ax)
ax.set_title("R² de clasificadores de densidad en datasets del Anexo")
ax.tick_params(axis="x", rotation=30)
fpath = img_dir / "anexo-density-clfs-r2.svg"
save_fig(fig, fpath)
show_generated()

## Datasets 15D: efecto de las dimensiones de ruido

Comparación de R² entre la versión 3D (sin ruido) y la versión 15D (con 12 dims
de ruido gaussiano) para los 4 datasets tridimensionales.

In [ ]:
datasets_3d_pairs = [
    ("pionono_0", "pionono_12"),
    ("eslabones_0", "eslabones_12"),
    ("helices_0", "helices_12"),
    ("hueveras_0", "hueveras_12"),
]

rows = []
for ds_3d, ds_15d in datasets_3d_pairs:
    for clf in bi.clf.unique():
        r2_3d = bi[(bi.dataset == ds_3d) & (bi.clf == clf)]["r2"].median()
        r2_15d = bi[(bi.dataset == ds_15d) & (bi.clf == clf)]["r2"].median()
        if pd.notna(r2_3d) and pd.notna(r2_15d):
            rows.append(
                {
                    "figura": ds_3d.split("_")[0],
                    "clf": clf,
                    "R² 3D": r2_3d,
                    "R² 15D": r2_15d,
                    "delta": r2_15d - r2_3d,
                }
            )

df_drop = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True, layout="tight")
for ax, figura in zip(
    axes, ["pionono", "eslabones", "helices", "hueveras"], strict=True
):
    sub = df_drop[df_drop.figura == figura].sort_values("delta")
    colors = [default_palette.get(c, "gray") for c in sub.clf]
    ax.barh(sub.clf, sub.delta, color=colors)
    ax.set_title(figura)
    ax.axvline(0, color="gray", linewidth=0.5)
    if ax == axes[0]:
        ax.set_xlabel("$\\Delta R^2$ (15D $-$ 3D)")

fig.suptitle("Caída de R² al agregar 12 dimensiones de ruido")
fpath = img_dir / "anexo-15d-delta-r2.svg"
save_fig(fig, fpath)
show_generated()

## Datasets multiclase: R² scatter fkdc vs kdc

Scatter de R² por semilla para fkdc vs kdc en iris, vino, pinguinos y anteojos.

In [ ]:
multik_datasets = ["iris", "vino", "pinguinos", "anteojos"]

fig, axes = plt.subplots(1, 4, figsize=(16, 4), layout="tight")
for ax, dataset in zip(axes, multik_datasets, strict=True):
    data_scatter = (
        bi[bi.dataset.eq(dataset) & bi.clf.isin(["fkdc", "kdc"])]
        .set_index(["semilla", "clf"])["r2"]
        .unstack()
    )
    if data_scatter.empty or "fkdc" not in data_scatter.columns:
        ax.set_title(f"{dataset} (sin datos)")
        continue
    data_scatter.plot(kind="scatter", y="fkdc", x="kdc", ax=ax)
    range_ = data_scatter.max().max() - data_scatter.min().min()
    x_left = data_scatter.min()["kdc"] - 0.1 * range_
    ax.set_xlim(x_left)
    ax.set_ylim(data_scatter.min()["fkdc"] - 0.1 * range_)
    ax.axline((x_left, x_left), slope=1, color="gray", linestyle="dotted")
    ax.set_title(dataset)

fig.suptitle("R² por semilla: fkdc vs kdc")
fpath = img_dir / "anexo-multik-fkdc-vs-kdc.svg"
save_fig(fig, fpath)
show_generated()

## Datasets de alta dimensión: digitos y mnist

R² scatter fkdc vs kdc para digitos y mnist.

In [ ]:
hd_datasets = ["digitos", "mnist"]

fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="tight")
for ax, dataset in zip(axes, hd_datasets, strict=True):
    data_scatter = (
        bi[bi.dataset.eq(dataset) & bi.clf.isin(["fkdc", "kdc"])]
        .set_index(["semilla", "clf"])["r2"]
        .unstack()
    )
    if data_scatter.empty or "fkdc" not in data_scatter.columns:
        ax.set_title(f"{dataset} (sin datos)")
        continue
    data_scatter.plot(kind="scatter", y="fkdc", x="kdc", ax=ax)
    range_ = data_scatter.max().max() - data_scatter.min().min()
    x_left = data_scatter.min()["kdc"] - 0.1 * range_
    ax.set_xlim(x_left)
    ax.set_ylim(data_scatter.min()["fkdc"] - 0.1 * range_)
    ax.axline((x_left, x_left), slope=1, color="gray", linestyle="dotted")
    ax.set_title(dataset)

fig.suptitle("R² por semilla: fkdc vs kdc")
fpath = img_dir / "anexo-hd-fkdc-vs-kdc.svg"
save_fig(fig, fpath)
show_generated()

## Resumen de archivos generados

### JSONs
- `{dataset}-r2-highlights.json` × 10 datasets

### SVGs
- `{dataset}-scatter.svg` × 10 datasets
- `{dataset}-r2-boxplot.svg` × 10 datasets
- `{dataset}-accuracy-boxplot.svg` × 10 datasets
- `anexo-density-clfs-r2.svg`
- `anexo-15d-delta-r2.svg`
- `anexo-multik-fkdc-vs-kdc.svg`
- `anexo-hd-fkdc-vs-kdc.svg`